In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout,Input
from tensorflow.keras.optimizers import Adam
import numpy as np
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)
from pathlib import Path
import matplotlib.pyplot as plt

from tensorflow.keras.models import load_model
import pandas as pd
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,    
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    classification_report
)
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from pathlib import Path
import sys

In [ ]:
# ============================================================
# MODEL 2 — LOAD SEQUENCES
# ============================================================
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

sequence_dir = PROJECT_ROOT / "data" / "sequences" / "model2_v2"

X_train_m2 = np.load(sequence_dir / "train_model2_X.npy")
y_train_m2 = np.load(sequence_dir / "train_model2_y.npy")

X_validation_m2 = np.load(
    sequence_dir / "validation_model2_X.npy"
)
y_validation_m2 = np.load(
    sequence_dir / "validation_model2_y.npy"
)

X_test_m2 = np.load(
    sequence_dir / "test_model2_X.npy"
)
y_test_m2 = np.load(
    sequence_dir / "test_model2_y.npy"
)

print("=" * 80)
print("MODEL 2 — SEQUENCES LOADED")
print("=" * 80)

print(f"X_train_m2:        {X_train_m2.shape}")
print(f"y_train_m2:        {y_train_m2.shape}")

print(f"X_validation_m2:   {X_validation_m2.shape}")
print(f"y_validation_m2:   {y_validation_m2.shape}")

print(f"X_test_m2:         {X_test_m2.shape}")
print(f"y_test_m2:         {y_test_m2.shape}")

In [ ]:
# ============================================================
# MODEL 2 — CLASS DEFINITIONS
# ============================================================

CLASS_NAMES = {
    0: "Contextual",
    1: "Ruido",
    2: "Desviacion de Correlacion",
    3: "Datos Faltantes",
    4: "Valores Fuera de Rango",
    5: "Sensor Atascado",
}

NUM_CLASSES = len(CLASS_NAMES)
SEQUENCE_LENGTH = X_train_m2.shape[1]
NUM_FEATURES = X_train_m2.shape[2]

print("Number of classes:", NUM_CLASSES)
print("Sequence length:", SEQUENCE_LENGTH)
print("Number of features:", NUM_FEATURES)

for class_id, class_name in CLASS_NAMES.items():
    print(f"{class_id}: {class_name}")

In [ ]:
# ============================================================
# MODEL 2 — CLASS WEIGHTS
# ============================================================

classes = np.unique(y_train_m2)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_m2
)

class_weights_m2 = {
    int(class_id): float(weight)
    for class_id, weight in zip(classes, class_weights_array)
}

print("=" * 80)
print("MODEL 2 — CLASS WEIGHTS")
print("=" * 80)

for class_id in classes:
    print(
        f"{class_id}: "
        f"{CLASS_NAMES[int(class_id)]:<30} "
        f"{class_weights_m2[int(class_id)]:.4f}"
    )

In [ ]:
# ============================================================
# MODEL 2 — MULTICLASS LSTM
# ============================================================

model2 = Sequential([
    
    Input(
        shape=(SEQUENCE_LENGTH, NUM_FEATURES)
    ),

    LSTM(
        64,
        return_sequences=True
    ),

    Dropout(0.2),

    LSTM(
        32,
        return_sequences=False
    ),

    Dropout(0.2),

    Dense(
        32,
        activation="relu"
    ),

    Dropout(0.2),

    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])


model2.summary()

In [ ]:
model2.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model 2 compiled successfully.")

In [ ]:
# ============================================================
# MODEL 2 — TRAINING CALLBACKS
# ============================================================
# ------------------------------------------------------------
# Output directories
# ------------------------------------------------------------
from tensorflow.keras import callbacks
model2_dir = PROJECT_ROOT / "models" / "model2_v2"
model2_dir.mkdir(parents=True, exist_ok=True)

best_model2_path = model2_dir / "lstm_model2_best.keras"


# ============================================================
# CUSTOM CALLBACK — VALIDATION MACRO F1
# ============================================================

class ValidationMacroF1(callbacks.Callback):

    def __init__(self, X_validation, y_validation):
        super().__init__()

        self.X_validation = X_validation
        self.y_validation = y_validation
        self.history = []

    def on_epoch_end(self, epoch, logs=None):

        logs = logs or {}

        # ----------------------------------------------------
        # Predict validation classes
        # ----------------------------------------------------

        probabilities = self.model.predict(
            self.X_validation,
            batch_size=256,
            verbose=0
        )

        predictions = np.argmax(
            probabilities,
            axis=1
        )

        # ----------------------------------------------------
        # Calculate macro F1
        # ----------------------------------------------------

        macro_f1 = f1_score(
            self.y_validation,
            predictions,
            average="macro",
            zero_division=0
        )

        # Store metric
        logs["val_macro_f1"] = macro_f1
        self.history.append(macro_f1)

        print(
            f" — val_macro_f1: {macro_f1:.4f}"
        )


# ============================================================
# CREATE CALLBACK
# ============================================================

macro_f1_callback = ValidationMacroF1(
    X_validation_m2,
    y_validation_m2
)

In [ ]:
# ============================================================
# MODEL 2 — CALLBACKS
# ============================================================

checkpoint_callback = ModelCheckpoint(
    filepath=str(best_model2_path),
    monitor="val_macro_f1",
    mode="max",
    save_best_only=True,
    verbose=1
)


early_stopping_callback = EarlyStopping(
    monitor="val_macro_f1",
    mode="max",
    patience=8,
    restore_best_weights=True,
    verbose=1
)


reduce_lr_callback = ReduceLROnPlateau(
    monitor="val_loss",
    mode="min",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [ ]:
# ============================================================
# MODEL 2 — TRAINING
# ============================================================

EPOCHS = 50
BATCH_SIZE = 256


history_model2 = model2.fit(
    X_train_m2,
    y_train_m2,

    validation_data=(
        X_validation_m2,
        y_validation_m2
    ),

    epochs=EPOCHS,
    batch_size=BATCH_SIZE,

    class_weight=class_weights_m2,

    callbacks=[
        macro_f1_callback,
        checkpoint_callback,
        early_stopping_callback,
        reduce_lr_callback
    ],

    verbose=1
)

In [ ]:
# ============================================================
# LOAD BEST MODEL 2
# ============================================================

best_model2 = load_model(
    best_model2_path
)

print("=" * 80)
print("BEST MODEL 2 LOADED")
print("=" * 80)

print("Path:", best_model2_path)

print("\nModel input shape:")
print(best_model2.input_shape)

print("\nModel output shape:")
print(best_model2.output_shape)

In [ ]:
# ============================================================
# BEST VALIDATION MACRO F1
# ============================================================

val_macro_f1 = history_model2.history["val_macro_f1"]

best_epoch = np.argmax(val_macro_f1)
best_f1 = val_macro_f1[best_epoch]

print("=" * 80)
print("BEST MODEL 2 VALIDATION RESULT")
print("=" * 80)

print(f"Best epoch:        {best_epoch + 1}")
print(f"Best val macro F1: {best_f1:.4f}")

In [ ]:
# ============================================================
# MODEL 2 — VALIDATION PREDICTIONS
# ============================================================

y_validation_proba = best_model2.predict(
    X_validation_m2,
    batch_size=256,
    verbose=1
)

y_validation_pred = np.argmax(
    y_validation_proba,
    axis=1
)

print("Validation predictions generated.")
print(f"Probability shape: {y_validation_proba.shape}")
print(f"Prediction shape:  {y_validation_pred.shape}")

In [ ]:
# ============================================================
# MODEL 2 — CLASS NAMES
# ============================================================

CLASS_NAMES = [
    "Contextual",
    "Ruido",
    "Desviacion de Correlacion",
    "Datos Faltantes",
    "Valores Fuera de Rango",
    "Sensor Atascado"
]

In [ ]:
print("=" * 100)
print("MODEL 2 — VALIDATION CLASSIFICATION REPORT")
print("=" * 100)

print(
    classification_report(
        y_validation_m2,
        y_validation_pred,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# MODEL 2 — VALIDATION F1 SCORES
# ============================================================

validation_macro_f1 = f1_score(
    y_validation_m2,
    y_validation_pred,
    average="macro",
    zero_division=0
)

validation_weighted_f1 = f1_score(
    y_validation_m2,
    y_validation_pred,
    average="weighted",
    zero_division=0
)

print("=" * 100)
print("MODEL 2 — VALIDATION F1 SCORES")
print("=" * 100)

print(f"Macro-F1:    {validation_macro_f1:.4f}")
print(f"Weighted-F1: {validation_weighted_f1:.4f}")

In [ ]:
# ============================================================
# MODEL 2 — VALIDATION CONFUSION MATRIX
# ============================================================

from sklearn.metrics import confusion_matrix

cm_validation = confusion_matrix(
    y_validation_m2,
    y_validation_pred
)

print("=" * 100)
print("MODEL 2 — VALIDATION CONFUSION MATRIX")
print("=" * 100)

print(cm_validation)

In [ ]:
# ============================================================
# MODEL 2 — VALIDATION CONFUSION MATRIX PLOT
# ============================================================

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(10, 8))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_validation,
    display_labels=CLASS_NAMES
)

disp.plot(
    ax=ax,
    xticks_rotation=45,
    values_format="d"
)

ax.set_title("Model 2 — Validation Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# MODEL 2 — PER-CLASS VALIDATION PERFORMANCE
# ============================================================

from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    y_validation_m2,
    y_validation_pred,
    labels=np.arange(NUM_CLASSES),
    zero_division=0
)

validation_results = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "Support": support
})

print(validation_results.to_string(index=False))

In [ ]:

# ============================================================
# MODEL 2 — TEST SET PREDICTIONS
# ============================================================

best_model2_path = PROJECT_ROOT / "models" / "model2_v2" / "lstm_model2_best.keras"

best_model2 = tf.keras.models.load_model(best_model2_path)

print("=" * 80)
print("MODEL 2 — TEST SET PREDICTIONS")
print("=" * 80)

print("X_test_m2 shape:", X_test_m2.shape)
print("y_test_m2 shape:", y_test_m2.shape)

test_probabilities_m2 = best_model2.predict(
    X_test_m2,
    batch_size=256,
    verbose=1
)

test_predictions_m2 = np.argmax(
    test_probabilities_m2,
    axis=1
)

print("\nPredictions completed.")
print("Probability matrix:", test_probabilities_m2.shape)
print("Predictions:", test_predictions_m2.shape)

In [ ]:

# ============================================================
# MODEL 2 — TEST SET EVALUATION
# ============================================================

CLASS_NAMES_M2 = [
    "Contextual",
    "Ruido",
    "Desviacion de Correlacion",
    "Datos Faltantes",
    "Valores Fuera de Rango",
    "Sensor Atascado"
]

print("=" * 100)
print("MODEL 2 — TEST CLASSIFICATION REPORT")
print("=" * 100)

print(
    classification_report(
        y_test_m2,
        test_predictions_m2,
        labels=np.arange(len(CLASS_NAMES_M2)),
        target_names=CLASS_NAMES_M2,
        digits=4,
        zero_division=0
    )
)

print("\n" + "=" * 100)
print("MODEL 2 — TEST CONFUSION MATRIX")
print("=" * 100)

cm_test_m2 = confusion_matrix(
    y_test_m2,
    test_predictions_m2,
    labels=np.arange(len(CLASS_NAMES_M2))
)

print("Rows = Actual class | Columns = Predicted class")
print("Class order:", CLASS_NAMES_M2)
print(cm_test_m2)

print("\nTest accuracy:", round(
    accuracy_score(y_test_m2, test_predictions_m2), 4
))

print("Test macro-F1:", round(
    f1_score(
        y_test_m2,
        test_predictions_m2,
        average="macro",
        zero_division=0
    ), 4
))

## here we have a problem, Sensor atascado has a bad recall results we have to fix it . but first let's analyse the results

In [ ]:
# ============================================================
# LOAD MODEL 2 FEATURE NAMES
# ============================================================

import json

scaled_dir = PROJECT_ROOT / "data" / "scaled" / "model2"

with open(scaled_dir / "model2_features.json", "r", encoding="utf-8") as f:
    MODEL2_FINAL_FEATURES = json.load(f)

print(f"Number of Model 2 features: {len(MODEL2_FINAL_FEATURES)}")

In [ ]:
# ============================================================
# MODEL 2 — PROBLEMATIC CLASS FEATURE COMPARISON
# ============================================================

problem_classes = {
    1: "Ruido",
    2: "Desviacion de Correlacion",
    5: "Sensor Atascado"
}

comparison_rows = []

for class_id, class_name in problem_classes.items():

    mask = y_validation_m2 == class_id

    class_mean = X_validation_target[mask].mean(axis=0)

    row = {
        "Class": class_name,
        "Samples": int(mask.sum())
    }

    for feature_idx, feature_name in enumerate(MODEL2_FINAL_FEATURES):
        row[feature_name] = class_mean[feature_idx]

    comparison_rows.append(row)

diagnostic_comparison = pd.DataFrame(comparison_rows)

diagnostic_comparison

In [ ]:

# ============================================================
# MODEL 2 — VALIDATION FEATURE ANALYSIS
# PROBLEMATIC CLASSES
# ============================================================

# The final timestep corresponds to the target row
X_validation_target = X_validation_m2[:, -1, :]

print("=" * 80)
print("MODEL 2 — TARGET ROW ANALYSIS")
print("=" * 80)

print(f"Target-row shape: {X_validation_target.shape}")
print(f"Labels shape:     {y_validation_m2.shape}")

# Class definitions
CLASS_NAMES = [
    "Contextual",
    "Ruido",
    "Desviacion de Correlacion",
    "Datos Faltantes",
    "Valores Fuera de Rango",
    "Sensor Atascado"
]

# Classes we want to investigate
problem_class_ids = [1, 2, 5]

# ------------------------------------------------------------
# Calculate mean value for every feature for each class
# ------------------------------------------------------------

class_means = {}

for class_id in problem_class_ids:

    mask = y_validation_m2 == class_id

    class_means[class_id] = X_validation_target[mask].mean(axis=0)

    print(
        f"{CLASS_NAMES[class_id]:30s}: "
        f"{mask.sum():4d} samples"
    )

# ------------------------------------------------------------
# Build comparison table
# ------------------------------------------------------------

feature_comparison = pd.DataFrame({
    "Feature": MODEL2_FINAL_FEATURES,
    "Ruido": class_means[1],
    "Desviacion": class_means[2],
    "Sensor_Atascado": class_means[5],
})

# Difference between highest and lowest class mean
feature_comparison["Max_difference"] = (
    feature_comparison[
        ["Ruido", "Desviacion", "Sensor_Atascado"]
    ].max(axis=1)
    -
    feature_comparison[
        ["Ruido", "Desviacion", "Sensor_Atascado"]
    ].min(axis=1)
)

# Sort by largest separation
feature_comparison = feature_comparison.sort_values(
    "Max_difference",
    ascending=False
)

print()
print("=" * 80)
print("TOP 30 FEATURES — CLASS SEPARATION")
print("=" * 80)

print(
    feature_comparison
    .head(30)
    .to_string(index=False)
)



In [ ]:
# ============================================================
# MODEL 2 — SENSOR ATASCADO ERROR ANALYSIS
# ============================================================

# Actual Sensor Atascado
stuck_mask = y_validation_m2 == 5

# Correctly predicted Sensor Atascado
stuck_correct_mask = stuck_mask & (y_validation_pred == 5)

# Sensor Atascado predicted as Ruido
stuck_as_ruido_mask = stuck_mask & (y_validation_pred == 1)

# Sensor Atascado predicted as Desviacion
stuck_as_desviacion_mask = stuck_mask & (y_validation_pred == 2)

print("=" * 80)
print("SENSOR ATASCADO — ERROR BREAKDOWN")
print("=" * 80)

print(f"Actual Sensor Atascado:              {stuck_mask.sum()}")
print(f"Correctly predicted:                 {stuck_correct_mask.sum()}")
print(f"Predicted as Ruido:                  {stuck_as_ruido_mask.sum()}")
print(f"Predicted as Desviacion:             {stuck_as_desviacion_mask.sum()}")

In [ ]:
# ============================================================
# SENSOR ATASCADO — FEATURES ASSOCIATED WITH ERRORS
# ============================================================

correct_values = X_validation_target[stuck_correct_mask]
ruido_values = X_validation_target[stuck_as_ruido_mask]
desviacion_values = X_validation_target[stuck_as_desviacion_mask]

error_feature_analysis = []

for feature_idx, feature_name in enumerate(MODEL2_FINAL_FEATURES):

    correct_mean = correct_values[:, feature_idx].mean()
    ruido_mean = ruido_values[:, feature_idx].mean()
    desviacion_mean = desviacion_values[:, feature_idx].mean()

    error_feature_analysis.append({
        "Feature": feature_name,
        "Correct_Stuck": correct_mean,
        "Stuck_as_Ruido": ruido_mean,
        "Stuck_as_Desviacion": desviacion_mean,
        "Max_difference": max(
            correct_mean,
            ruido_mean,
            desviacion_mean
        ) - min(
            correct_mean,
            ruido_mean,
            desviacion_mean
        )
    })

error_feature_analysis_df = (
    pd.DataFrame(error_feature_analysis)
    .sort_values("Max_difference", ascending=False)
)

print("=" * 100)
print("TOP FEATURES — CORRECT STUCK VS MISCLASSIFIED STUCK")
print("=" * 100)

print(
    error_feature_analysis_df
    .head(30)
    .to_string(index=False)
)

In [ ]:
# ============================================================
# SENSOR ATASCADO — PREDICTION CONFIDENCE ANALYSIS
# ============================================================

stuck_error_mask = (
    (y_validation_m2 == 5) &
    (y_validation_pred != 5)
)

stuck_error_indices = np.where(stuck_error_mask)[0]

stuck_error_proba = y_validation_proba[stuck_error_indices]
stuck_error_pred = y_validation_pred[stuck_error_indices]

# Probability assigned to the true class: Sensor Atascado
true_class_probability = stuck_error_proba[:, 5]

# Probability assigned to the predicted class
predicted_class_probability = np.max(
    stuck_error_proba,
    axis=1
)

confidence_summary = pd.DataFrame({
    "True_Class_Probability": true_class_probability,
    "Predicted_Class_Probability": predicted_class_probability,
    "Predicted_Class": stuck_error_pred
})

print("=" * 100)
print("SENSOR ATASCADO — MISCLASSIFICATION CONFIDENCE")
print("=" * 100)

print(
    confidence_summary[
        [
            "True_Class_Probability",
            "Predicted_Class_Probability"
        ]
    ].describe()
)

print()
print("Predicted classes:")
print(
    confidence_summary["Predicted_Class"]
    .value_counts()
    .rename(index={
        1: "Ruido",
        2: "Desviacion de Correlacion"
    })
)

In [ ]:
# ============================================================
# SENSOR ATASCADO — CONFIDENCE DISTRIBUTION
# ============================================================

print("=" * 100)
print("TRUE SENSOR ATASCADO PROBABILITY")
print("=" * 100)

for threshold in [0.10, 0.20, 0.30, 0.40, 0.50]:

    percentage = (
        true_class_probability >= threshold
    ).mean() * 100

    print(
        f"P(Sensor Atascado) >= {threshold:.2f}: "
        f"{percentage:.2f}%"
    )

In [ ]:
# ============================================================
# MODEL 2 — SENSOR ATASCADO TRAIN VS VALIDATION
# ============================================================

X_train_target = X_train_m2[:, -1, :]
X_validation_target = X_validation_m2[:, -1, :]

train_stuck_mask = y_train_m2 == 5
validation_stuck_mask = y_validation_m2 == 5

print("=" * 100)
print("SENSOR ATASCADO — TRAIN VS VALIDATION")
print("=" * 100)

print(f"Train Sensor Atascado:      {train_stuck_mask.sum()}")
print(f"Validation Sensor Atascado: {validation_stuck_mask.sum()}")

In [ ]:
# ============================================================
# SENSOR ATASCADO — TRAIN VS VALIDATION FEATURE COMPARISON
# ============================================================

train_stuck_means = X_train_target[train_stuck_mask].mean(axis=0)
validation_stuck_means = X_validation_target[validation_stuck_mask].mean(axis=0)

train_validation_comparison = pd.DataFrame({
    "Feature": MODEL2_FINAL_FEATURES,
    "Train": train_stuck_means,
    "Validation": validation_stuck_means
})

train_validation_comparison["Absolute_Difference"] = (
    train_validation_comparison["Train"]
    -
    train_validation_comparison["Validation"]
).abs()

train_validation_comparison = (
    train_validation_comparison
    .sort_values("Absolute_Difference", ascending=False)
)

print("=" * 100)
print("TOP 30 TRAIN / VALIDATION DIFFERENCES — SENSOR ATASCADO")
print("=" * 100)

print(
    train_validation_comparison
    .head(30)
    .to_string(index=False)
)

In [ ]:
print("Model 2 sequence feature count:", X_train_m2.shape[-1])
print("Number of features:", len(MODEL2_FINAL_FEATURES))

print("\nPotential sensor-specific features:")
for feature in MODEL2_FINAL_FEATURES:
    if any(sensor in feature for sensor in [
        "PCO2EXT", "PHEXT", "PRAD", "PRGINT", "PTEXT",
        "PVV", "XCO2I", "XHINV", "XTINV"
    ]):
        print(feature)

In [ ]:
# ============================================================
# SENSOR ATASCADO — TRAIN VS VALIDATION SEQUENCE STATISTICS
# ============================================================

train_stuck = X_train_m2[y_train_m2 == 5]
validation_stuck = X_validation_m2[y_validation_m2 == 5]

# Mean over the 20 timesteps
train_stuck_seq_mean = train_stuck.mean(axis=1)
validation_stuck_seq_mean = validation_stuck.mean(axis=1)

# Standard deviation over the 20 timesteps
train_stuck_seq_std = train_stuck.std(axis=1)
validation_stuck_seq_std = validation_stuck.std(axis=1)

train_seq_mean = train_stuck_seq_mean.mean(axis=0)
validation_seq_mean = validation_stuck_seq_mean.mean(axis=0)

train_seq_std = train_stuck_seq_std.mean(axis=0)
validation_seq_std = validation_stuck_seq_std.mean(axis=0)

sequence_comparison = pd.DataFrame({
    "Feature": MODEL2_FINAL_FEATURES,
    "Train_Mean": train_seq_mean,
    "Validation_Mean": validation_seq_mean,
    "Mean_Difference": np.abs(
        train_seq_mean - validation_seq_mean
    ),
    "Train_Temporal_STD": train_seq_std,
    "Validation_Temporal_STD": validation_seq_std,
    "Temporal_STD_Difference": np.abs(
        train_seq_std - validation_seq_std
    )
})

sequence_comparison = sequence_comparison.sort_values(
    "Mean_Difference",
    ascending=False
)

print("=" * 100)
print("TOP 30 SEQUENCE-LEVEL TRAIN / VALIDATION DIFFERENCES")
print("=" * 100)

print(
    sequence_comparison
    .head(30)
    .to_string(index=False)
)

In [ ]:
# ============================================================
# MODEL 2 — SENSOR ATASCADO TIME-OF-DAY DISTRIBUTION
# ============================================================

train_stuck_idx = np.where(y_train_m2 == 5)[0]
validation_stuck_idx = np.where(y_validation_m2 == 5)[0]

hour_idx = MODEL2_FINAL_FEATURES.index("hour_sin")
hour_cos_idx = MODEL2_FINAL_FEATURES.index("hour_cos")

train_hour_sin = X_train_m2[train_stuck_idx, -1, hour_idx]
train_hour_cos = X_train_m2[train_stuck_idx, -1, hour_cos_idx]

validation_hour_sin = X_validation_m2[
    validation_stuck_idx, -1, hour_idx
]

validation_hour_cos = X_validation_m2[
    validation_stuck_idx, -1, hour_cos_idx
]

# Recover hour approximately from sin/cos.
train_hours = (
    np.arctan2(train_hour_sin, train_hour_cos) * 24 / (2 * np.pi)
) % 24

validation_hours = (
    np.arctan2(validation_hour_sin, validation_hour_cos) * 24 / (2 * np.pi)
) % 24

print("=" * 100)
print("SENSOR ATASCADO — TIME OF DAY")
print("=" * 100)

print(f"Train mean hour:       {train_hours.mean():.2f}")
print(f"Validation mean hour:  {validation_hours.mean():.2f}")

print(f"\nTrain median hour:      {np.median(train_hours):.2f}")
print(f"Validation median hour: {np.median(validation_hours):.2f}")

In [ ]:
def time_period(hour):
    if hour < 6:
        return "Night"
    elif hour < 12:
        return "Morning"
    elif hour < 18:
        return "Afternoon"
    else:
        return "Evening"


train_periods = pd.Series(
    [time_period(h) for h in train_hours]
).value_counts(normalize=True)

validation_periods = pd.Series(
    [time_period(h) for h in validation_hours]
).value_counts(normalize=True)

time_comparison = pd.DataFrame({
    "Train": train_periods,
    "Validation": validation_periods
}).fillna(0)

time_comparison["Absolute_Difference"] = (
    time_comparison["Train"] -
    time_comparison["Validation"]
).abs()

print("\n" + "=" * 100)
print("SENSOR ATASCADO — TIME PERIOD DISTRIBUTION")
print("=" * 100)

print(
    time_comparison
    .sort_values("Absolute_Difference", ascending=False)
    .to_string()
)

In [ ]:
# ============================================================
# MODEL 2 — INSPECT ANOMALY TRACEABILITY COLUMNS
# ============================================================

train_model2_path = PROJECT_ROOT / "data" / "model2" / "train_model2.parquet"
validation_model2_path = PROJECT_ROOT / "data" / "model2" / "validation_model2.parquet"

train_model2_diag = pd.read_parquet(train_model2_path)
validation_model2_diag = pd.read_parquet(validation_model2_path)

print("Train shape:", train_model2_diag.shape)
print("Validation shape:", validation_model2_diag.shape)

print("\nColumns related to anomaly traceability:")
for col in train_model2_diag.columns:
    if any(
        keyword in col.lower()
        for keyword in [
            "anom",
            "sensor",
            "etiqueta",
            "tipo",
            "target"
        ]
    ):
        print(col)

In [ ]:
# ============================================================
# MODEL 2 — SENSOR ATASCADO BY AFFECTED SENSOR
# ============================================================

train_stuck_diag = train_model2_diag[
    train_model2_diag["tipo_anomalia"] == "Sensor Atascado"
].copy()

validation_stuck_diag = validation_model2_diag[
    validation_model2_diag["tipo_anomalia"] == "Sensor Atascado"
].copy()

print("=" * 100)
print("SENSOR ATASCADO — AFFECTED SENSOR DISTRIBUTION")
print("=" * 100)

print(f"\nTrain Sensor Atascado rows:      {len(train_stuck_diag)}")
print(f"Validation Sensor Atascado rows: {len(validation_stuck_diag)}")

train_sensor_dist = (
    train_stuck_diag["sensor_anomalia"]
    .value_counts(normalize=True)
    .rename("Train")
)

validation_sensor_dist = (
    validation_stuck_diag["sensor_anomalia"]
    .value_counts(normalize=True)
    .rename("Validation")
)

sensor_distribution = pd.concat(
    [train_sensor_dist, validation_sensor_dist],
    axis=1
).fillna(0)

sensor_distribution["Absolute_Difference"] = (
    sensor_distribution["Train"]
    - sensor_distribution["Validation"]
).abs()

sensor_distribution = sensor_distribution.sort_values(
    "Absolute_Difference",
    ascending=False
)

print("\n")
print(
    sensor_distribution.to_string(
        formatters={
            "Train": "{:.4f}".format,
            "Validation": "{:.4f}".format,
            "Absolute_Difference": "{:.4f}".format
        }
    )
)